# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
if not os.path.exists(REPO_ROOT):
    REPO_ROOT = "."
    OUT_DIR = "."

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}, Device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chuẩn bị dữ liệu Part 0
processed_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed_dir}/train.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# Kiểm tra trung bình và độ lệch chuẩn của 10 cột số trên X_tr
mean_10 = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
std_10 = data["X_tr"][:, :10].std(dim=0).cpu().numpy()
print("X_tr 10 numeric features mean (~0):", np.round(mean_10, 4))
print("X_tr 10 numeric features std  (~1):", np.round(std_10, 4))
assert abs(mean_10).max() < 1e-4, "Mean chưa chuẩn hoá về ~0"
assert abs(std_10 - 1.0).max() < 1e-2, "Std chưa chuẩn hoá về ~1"
assert len(data["eval_row_id"]) == len(data["X_eval"]) == 116203, "eval_row_id không khớp shape eval"
print("Part 0 verification PASSED!")


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# ===== Part 1 — Model và kiểm tra "sức khoẻ" ban đầu =====
import math
import torch
import torch.nn.functional as F

# 1. Khởi tạo M-base và assert số tham số
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"M-base total parameters: {n_params}")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Sai số tham số: {n_params}"

# 2. Thử lô giả lập (8, 54) và kiểm tra shape
x_dummy = torch.randn(8, 54, device=device)
logits = model(x_dummy)
print(f"Dummy batch output shape: {logits.shape}")
assert logits.shape == (8, 7), f"Sai shape đầu ra: {logits.shape}"

# 3. Đo loss bước 0 trên Val ở chế độ eval()
model.eval()
with torch.no_grad():
    val_logits = model(data["X_val"])
    loss_0 = F.cross_entropy(val_logits, data["y_val"]).item()
theo_loss = math.log(7)
print(f"Loss bước 0 trên Val: {loss_0:.4f} (Lý thuyết ln(7) = {theo_loss:.4f}, lệch: {abs(loss_0 - theo_loss):.4f})")

# 4. Quá khớp 20 mẫu (Overfit 20 samples)
x_sub = data["X_tr"][:20]
y_sub = data["y_tr"][:20]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_model.train()
optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-2)

for step in range(250):
    optimizer.zero_grad()
    out = overfit_model(x_sub)
    loss = F.cross_entropy(out, y_sub)
    loss.backward()
    optimizer.step()

overfit_model.eval()
with torch.no_grad():
    final_logits = overfit_model(x_sub)
    final_loss = F.cross_entropy(final_logits, y_sub).item()
    acc = (final_logits.argmax(dim=-1) == y_sub).float().mean().item()

print(f"Overfit 20 mẫu: Loss = {final_loss:.6f} | Accuracy = {acc * 100:.1f}%")
assert final_loss < 0.05, f"Loss chưa về gần 0: {final_loss}"
assert acc == 1.0, f"Accuracy chưa đạt 100%: {acc}"

# 5. Kiểm tra dòng chảy gradient sau backward()
model.train()
model.zero_grad()
sample_loss = F.cross_entropy(model(data["X_tr"][:32]), data["y_tr"][:32])
sample_loss.backward()

print("Kiểm tra gradient:")
for name, param in model.named_parameters():
    assert param.grad is not None, f"Tham số {name} bị None gradient!"
    norm = param.grad.norm().item()
    assert norm > 0.0, f"Tham số {name} bị gradient 0.0!"
    print(f"  {name:25s} | grad_norm: {norm:.6f}")

print("\n=> Part 1 kiểm tra sức khoẻ PASSED 100%!")


**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary